### Loading the triage_200 csv datafile



In [3]:
import pandas as pd
df = pd.read_csv(r"C:\Users\Deepalakshmi\Downloads\sample_emails_with_triage_200.csv")
df.head()

,id,sender,subject,body,priority,triage_label
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond


In [4]:
#select 100 test emails
eval_df = df.sample(100,random_state=42)
eval_df = eval_df.reset_index(drop=True)
eval_df.head()

,id,sender,subject,body,priority,triage_label
0,96,no-reply@service.com,Weekly Newsletter,Your order #3634 has been shipped and is expec...,low,ignore
1,16,news@techblog.com,Payment Overdue,"Hi, don't miss our sale with discounts up to 7...",low,notify_human
2,31,news@techblog.com,Weekly Newsletter,Notice: Your account will be locked unless ver...,high,notify_human
3,159,no-reply@service.com,Welcome to Service,Reminder: The client meeting is scheduled at 9...,low,respond
4,129,sales@shop.com,Invoice Due,Your order #6464 has been shipped and is expec...,low,respond


### Creating Ideal_response column

In [5]:
columns = ["id","sender","subject","body","priority","triage_label"]
df = pd.DataFrame(df, columns=columns)

# Function using if-elif-else
def ideal_response(label):
    if label == "notify_human":
        return "Escalate to human support"
    elif label == "respond":
        return "Send automated response"
    elif label == "ignore":
        return "No action required"
    else:
        return "Manual review needed"

# Create new column
df["ideal_response"] = df["triage_label"].apply(ideal_response)

df[["triage_label","ideal_response"]].head()


,triage_label,ideal_response
0,notify_human,Escalate to human support
1,respond,Send automated response
2,ignore,No action required
3,respond,Send automated response
4,respond,Send automated response


# Milestone2_Lingaeswari

### Loading Evaluation_sample dataset


In [6]:
import pandas as pd
df = pd.read_csv(r"C:\Users\Deepalakshmi\Downloads\email_evaluation_dataset_Lingaeswari.csv")

In [7]:
df.head()

,id,email_text,expected_action,expected_tone
0,1,This is a reminder to attend the client discus...,notify,neutral
1,2,Please remember the project review meeting hap...,notify,neutral
2,3,This is a reminder to attend the client discus...,notify,neutral
3,4,This is a reminder to attend the weekly update...,notify,neutral
4,5,This is a reminder to attend the team sync mee...,notify,neutral


### Email assistant function

In [9]:
def email_assistant(email_text):
    text = email_text.lower()
    
    if "urgent" in text or "submit" in text or "deadline" in text:
        return "notify", "urgent"
    
    elif "thank you" in text:
        return "ignore", "polite"
    
    else:
        return "review", "neutral"

    
    

### 1. Applying assistant function to the dataset

In [16]:
# Apply assistant logic to each email
df["prediction"] = df["email_text"].apply(email_assistant)
df["prediction"]


0     (review, neutral)
1     (review, neutral)
2     (review, neutral)
3     (review, neutral)
4     (review, neutral)
            ...        
95     (ignore, polite)
96    (review, neutral)
97     (ignore, polite)
98     (ignore, polite)
99     (ignore, polite)
Name: prediction, Length: 100, dtype: object

### 2. Creating separate output columns

In [17]:
df[["predicted_action", "predicted_tone"]] = pd.DataFrame(
    df["prediction"].tolist(), index=df.index
)



In [23]:
df[["predicted_action", "predicted_tone"]] 

,predicted_action,predicted_tone
0,review,neutral
1,review,neutral
2,review,neutral
3,review,neutral
4,review,neutral
...,...,...
95,ignore,polite
96,review,neutral
97,ignore,polite
98,ignore,polite


### 3. Compare predictions with expected values

In [21]:
df["action_correct"] = df["predicted_action"] == df["expected_action"]
df["tone_correct"] = df["predicted_tone"] == df["expected_tone"]
df.head()

,id,email_text,expected_action,expected_tone,predicted_action,predicted_tone,prediction,action_correct,tone_correct
0,1,This is a reminder to attend the client discus...,notify,neutral,review,neutral,"(review, neutral)",False,True
1,2,Please remember the project review meeting hap...,notify,neutral,review,neutral,"(review, neutral)",False,True
2,3,This is a reminder to attend the client discus...,notify,neutral,review,neutral,"(review, neutral)",False,True
3,4,This is a reminder to attend the weekly update...,notify,neutral,review,neutral,"(review, neutral)",False,True
4,5,This is a reminder to attend the team sync mee...,notify,neutral,review,neutral,"(review, neutral)",False,True


### 4. Calculating accuracy

In [24]:
action_accuracy = df["action_correct"].mean() * 100
tone_accuracy = df["tone_correct"].mean() * 100

print(f"Action Accuracy: {action_accuracy:.2f}%")
print(f"Tone Accuracy: {tone_accuracy:.2f}%")


Action Accuracy: 10.00%
Tone Accuracy: 30.00%


### 5. Perform error analysis

#### Action errors

In [25]:
action_errors = df[df["action_correct"] == False][
    ["email_text", "expected_action", "predicted_action"]
]
action_errors


,email_text,expected_action,predicted_action
0,This is a reminder to attend the client discus...,notify,review
1,Please remember the project review meeting hap...,notify,review
2,This is a reminder to attend the client discus...,notify,review
3,This is a reminder to attend the weekly update...,notify,review
4,This is a reminder to attend the team sync mee...,notify,review
...,...,...,...
90,This email is to inform you about recent updat...,ignore,review
91,This email is to inform you about recent updat...,ignore,review
92,This email is to inform you about recent updat...,ignore,review
94,This email is to inform you about recent updat...,ignore,review


#### Tone errors

In [26]:
tone_errors = df[df["tone_correct"] == False][
    ["email_text", "expected_tone", "predicted_tone"]
]
tone_errors


,email_text,expected_tone,predicted_tone
10,Payment reminder: Your invoice INV-2000 amount...,urgent,neutral
11,"Dear Customer, invoice INV-2001 for INR 6500 i...",urgent,neutral
12,Payment reminder: Your invoice INV-2002 amount...,urgent,neutral
13,"Dear Customer, invoice INV-2003 for INR 6500 i...",urgent,neutral
14,"Dear Customer, invoice INV-2004 for INR 6500 i...",urgent,neutral
...,...,...,...
93,Thank you for participating in the event. This...,neutral,polite
95,Thank you for participating in the event. This...,neutral,polite
97,Thank you for participating in the event. This...,neutral,polite
98,Thank you for participating in the event. This...,neutral,polite


### 6. Save your output CSV

In [29]:
df.to_csv(r"C:\Users\Deepalakshmi\Downloads\data\milestone2_output_lingaeswari.csv", index=False)
